# Cross-Asset Momentum Strategy v8 — Refined

**Design philosophy:** v6's always-invested rank-weighted core, enhanced with
signal-based weight adjustments per professor's Week 11 feedback.

Per Man Group/Winton (Slide 18): 'They scale down, not just exit.
Exit is continuous, not discrete.'

**Three signal tiers (applied to base rank-weight):**
- **STRONG:** top 5 rank + EMA confirmed → 1.3x boost
- **NORMAL:** EMA confirmed, not top 5 → 1.0x (base weight)
- **WEAKENING:** EMA not confirmed → 0.5x penalty

**Risk controls:**
- Max single-asset weight: 25% (prevents over-concentration)
- Graduated regime filter: 96% / 94% / 92% invested (from v6)
- 2% turnover buffer (prevents daily whipsaw)

**Exit strategy:** When momentum decays → rank drops → weight drops automatically.
When EMA breaks → 0.5x penalty halves weight immediately.
Capital continuously flows from weakening to strengthening assets.
This IS the exit — continuous, not discrete.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

UNIVERSE = {
    'GLD': 'Gold', 'SLV': 'Silver', 'CPER': 'Copper', 'DBA': 'Agriculture',
    'XLE': 'Energy Sector', 'XLK': 'Technology Sector', 'XLF': 'Financials Sector',
    'TLT': '20+ Yr Treasuries', 'LQD': 'IG Corp Bonds', 'HYG': 'High Yield Bonds', 'TIP': 'TIPS',
    'UUP': 'US Dollar Index', 'FXE': 'Euro', 'FXY': 'Japanese Yen',
}

CAPITAL = 50_000_000
N_ASSETS = len(UNIVERSE)

# Signal parameters
MOM_LOOKBACK = 252
MOM_SKIP = 21
EMA_SHORT = 10
EMA_LONG = 50

# Weight multipliers — 3 tiers only
MULT_STRONG = 1.3      # top 5 + EMA confirmed
MULT_NORMAL = 1.0      # EMA confirmed, not top 5
MULT_WEAKENING = 0.5   # EMA not confirmed

TOP_TIER = 5
MAX_WEIGHT = 0.25      # max 25% in any single asset
TURNOVER_BUFFER = 0.02 # don't trade if weight change < 2%

# Graduated regime filter (from v6)
REGIME_FULL = 0.96
REGIME_CAUTION = 0.94
REGIME_STRESS = 0.92

# Dates
START_DATE = '2012-01-01'
END_DATE = '2026-04-17'

print(f"Universe: {N_ASSETS} ETFs")
print(f"Multipliers: STRONG={MULT_STRONG}x, NORMAL={MULT_NORMAL}x, WEAKENING={MULT_WEAKENING}x")
print(f"Max single-asset weight: {MAX_WEIGHT:.0%}")
print(f"Turnover buffer: {TURNOVER_BUFFER:.0%}")
print(f"Regime: {REGIME_FULL:.0%} / {REGIME_CAUTION:.0%} / {REGIME_STRESS:.0%}")

## Step 1: Download Data

In [ ]:
tickers = list(UNIVERSE.keys())
raw_data = yf.download(tickers, start=START_DATE, end=END_DATE, progress=False)

if isinstance(raw_data.columns, pd.MultiIndex):
    col = 'Adj Close' if 'Adj Close' in raw_data.columns.get_level_values(0) else 'Close'
    prices = raw_data[col].copy()
else:
    prices = raw_data[['Adj Close']].copy() if 'Adj Close' in raw_data.columns else raw_data[['Close']].copy()

prices = prices.ffill(limit=5).dropna()
daily_returns = prices.pct_change()

print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"Trading days: {len(prices)}")

## Step 2: Compute Signals

In [ ]:
# Resample to weekly (Friday close) — same as v6
weekly_prices = prices.resample('W-FRI').last().dropna(how='all')
weekly_returns = weekly_prices.pct_change()

# 12-minus-1 month momentum on weekly data
# ~52 weeks lookback, skip ~4 weeks
price_past = weekly_prices.shift(4)
price_anchor = weekly_prices.shift(52)
momentum = (price_past / price_anchor) - 1

# EMA on daily data, sampled weekly
ema_short = prices.ewm(span=EMA_SHORT, adjust=False).mean()
ema_long = prices.ewm(span=EMA_LONG, adjust=False).mean()
ema_confirmed_daily = (ema_short > ema_long)
ema_confirmed = ema_confirmed_daily.resample('W-FRI').last()

# Ranks
mom_rank = momentum.rank(axis=1, method='average', ascending=True)

# Drop warmup
warmup_date = weekly_prices.index[52 + 2]
momentum = momentum.loc[warmup_date:]
ema_confirmed = ema_confirmed.loc[warmup_date:]
mom_rank = mom_rank.loc[warmup_date:]
weekly_returns = weekly_returns.loc[warmup_date:]

print(f"Ranking signal: 12-minus-1 month return (weekly)")
print(f"EMA filter: EMA({EMA_SHORT}) vs EMA({EMA_LONG})")
print(f"Weeks with signal: {len(momentum)}")

# Latest signals
print(f"\nLatest signals ({momentum.index[-1].date()}):")
latest_mom = momentum.iloc[-1].sort_values(ascending=False)
latest_ema = ema_confirmed.iloc[-1]
latest_rank = mom_rank.iloc[-1]
for i, (t, m) in enumerate(latest_mom.items(), 1):
    rank = int(latest_rank[t])
    ema_ok = latest_ema.get(t, False)
    if rank >= (N_ASSETS - TOP_TIER + 1) and ema_ok:
        tag = f'STRONG ({MULT_STRONG}x)'
    elif ema_ok:
        tag = f'NORMAL ({MULT_NORMAL}x)'
    else:
        tag = f'WEAKENING ({MULT_WEAKENING}x)'
    print(f"  {i:>2}. {t:5s} ({UNIVERSE[t]:20s}): {m:+.2%}  Rank {rank:>2}  [{tag}]")

## Step 3: Compute Weekly Weights

In [ ]:
# Regime filter: trailing 4-week returns of GLD and TLT
gld_trail = weekly_prices['GLD'].pct_change(4)
tlt_trail = weekly_prices['TLT'].pct_change(4)

regime_pct = pd.Series(REGIME_FULL, index=momentum.index)
for date in momentum.index:
    gld_r = gld_trail.get(date, 0)
    tlt_r = tlt_trail.get(date, 0)
    if pd.isna(gld_r): gld_r = 0
    if pd.isna(tlt_r): tlt_r = 0
    
    neg_count = (gld_r < 0) + (tlt_r < 0)
    if neg_count == 0:
        regime_pct[date] = REGIME_FULL
    elif neg_count == 1:
        regime_pct[date] = REGIME_CAUTION
    else:
        regime_pct[date] = REGIME_STRESS

# Base rank weights
rank_sums = mom_rank.sum(axis=1)
base_weights = mom_rank.div(rank_sums, axis=0)

# Apply multipliers (3 tiers)
multiplied = base_weights.copy()
for date in momentum.index:
    for ticker in tickers:
        rank = mom_rank.loc[date].get(ticker, 0)
        ema_ok = ema_confirmed.loc[date].get(ticker, False)
        
        if pd.isna(rank):
            multiplied.loc[date, ticker] = base_weights.loc[date, ticker] * MULT_WEAKENING
            continue
        
        if rank >= (N_ASSETS - TOP_TIER + 1) and ema_ok:
            multiplied.loc[date, ticker] = base_weights.loc[date, ticker] * MULT_STRONG
        elif ema_ok:
            multiplied.loc[date, ticker] = base_weights.loc[date, ticker] * MULT_NORMAL
        else:
            multiplied.loc[date, ticker] = base_weights.loc[date, ticker] * MULT_WEAKENING

# Normalize
mult_sums = multiplied.sum(axis=1)
target_weights = multiplied.div(mult_sums, axis=0)

# Apply max weight cap
def cap_weights(row, max_w=MAX_WEIGHT):
    capped = row.clip(upper=max_w)
    # Redistribute excess to uncapped positions proportionally
    excess = row.sum() - capped.sum()
    uncapped_mask = capped < max_w
    if uncapped_mask.sum() > 0 and excess > 0:
        uncapped_total = capped[uncapped_mask].sum()
        if uncapped_total > 0:
            capped[uncapped_mask] += excess * (capped[uncapped_mask] / uncapped_total)
    return capped

target_weights = target_weights.apply(cap_weights, axis=1)

# Apply regime filter
target_weights = target_weights.mul(regime_pct, axis=0)

# Lag by 1 week
weights = target_weights.shift(1).dropna(how='all')

# Apply turnover buffer
buffered = weights.copy()
for i in range(1, len(buffered)):
    prev = buffered.iloc[i-1]
    curr = weights.iloc[i]
    delta = (curr - prev).abs()
    buffered.iloc[i] = np.where(delta < TURNOVER_BUFFER, prev, curr)
    # Renormalize to regime-adjusted level
    row_sum = buffered.iloc[i].sum()
    target_invest = regime_pct.iloc[i] if i < len(regime_pct) else REGIME_FULL
    if row_sum > 0:
        buffered.iloc[i] = buffered.iloc[i] / row_sum * target_invest

weights = buffered

print(f"Weight computation complete: {len(weights)} weeks")
print(f"Weight range: {weights.min().min():.2%} to {weights.max().max():.2%}")
print(f"Sum range: {weights.sum(axis=1).min():.2%} to {weights.sum(axis=1).max():.2%}")

# Regime breakdown
total_weeks = len(regime_pct)
print(f"\nRegime breakdown:")
print(f"  Full ({REGIME_FULL:.0%}):    {(regime_pct == REGIME_FULL).sum()}/{total_weeks} ({(regime_pct == REGIME_FULL).mean():.0%})")
print(f"  Caution ({REGIME_CAUTION:.0%}): {(regime_pct == REGIME_CAUTION).sum()}/{total_weeks} ({(regime_pct == REGIME_CAUTION).mean():.0%})")
print(f"  Stress ({REGIME_STRESS:.0%}):  {(regime_pct == REGIME_STRESS).sum()}/{total_weeks} ({(regime_pct == REGIME_STRESS).mean():.0%})")

In [ ]:
# Latest weights
print(f"\nLatest weights ({weights.index[-1].date()}):")
latest_w = weights.iloc[-1].sort_values(ascending=False)
for t, w in latest_w.items():
    rank = int(latest_rank[t])
    ema_ok = latest_ema.get(t, False)
    if rank >= (N_ASSETS - TOP_TIER + 1) and ema_ok:
        tag = 'STRONG'
    elif ema_ok:
        tag = 'NORMAL'
    else:
        tag = 'WEAK'
    print(f"  {t:5s} ({UNIVERSE[t]:20s}): {w:.1%}  [{tag}]")

print(f"\nTotal invested: {latest_w.sum():.1%}")
print(f"Cash: {1 - latest_w.sum():.1%}")

## Step 4: Compute Returns

In [ ]:
# Weekly portfolio returns
common_idx = weights.index.intersection(weekly_returns.index)
w = weights.loc[common_idx]
r = weekly_returns.loc[common_idx]

portfolio_returns = (w * r).sum(axis=1)
portfolio_returns = portfolio_returns.dropna()
first_valid = (portfolio_returns != 0).idxmax()
portfolio_returns = portfolio_returns.loc[first_valid:]

cumulative = (1 + portfolio_returns).cumprod()

print(f"Backtest: {portfolio_returns.index[0].date()} to {portfolio_returns.index[-1].date()}")
print(f"Weeks: {len(portfolio_returns)}")
print(f"Cumulative return: {cumulative.iloc[-1] - 1:.2%}")

## Step 5: Benchmarks & Momentum Spread

In [ ]:
# Equal-weight benchmark
ew_returns = weekly_returns.loc[first_valid:].mean(axis=1)
cum_ew = (1 + ew_returns).cumprod()

# 60/40
try:
    bd = yf.download(['SPY', 'AGG'], start=START_DATE, end=END_DATE, progress=False)
    cn = 'Adj Close' if 'Adj Close' in bd.columns.get_level_values(0) else 'Close'
    spy_w = bd[cn]['SPY'].resample('W-FRI').last().pct_change()
    agg_w = bd[cn]['AGG'].resample('W-FRI').last().pct_change()
    bench_6040 = (0.6 * spy_w + 0.4 * agg_w).reindex(portfolio_returns.index).fillna(0)
    cum_6040 = (1 + bench_6040).cumprod()
    has_6040 = True
    print("60/40 SPY/AGG: loaded")
except:
    has_6040 = False

# Inverse momentum
inv_ranks = momentum.rank(axis=1, method='average', ascending=False)
inv_base = inv_ranks.div(inv_ranks.sum(axis=1), axis=0)

# Apply SAME multiplier logic to inverse (3 tiers based on inverse ranking)
inv_multiplied = inv_base.copy()
for date in momentum.index:
    for ticker in tickers:
        inv_rank = inv_ranks.loc[date].get(ticker, 0)
        ema_ok = ema_confirmed.loc[date].get(ticker, False)
        if pd.isna(inv_rank):
            inv_multiplied.loc[date, ticker] = inv_base.loc[date, ticker] * MULT_WEAKENING
            continue
        if inv_rank >= (N_ASSETS - TOP_TIER + 1) and ema_ok:
            inv_multiplied.loc[date, ticker] = inv_base.loc[date, ticker] * MULT_STRONG
        elif ema_ok:
            inv_multiplied.loc[date, ticker] = inv_base.loc[date, ticker] * MULT_NORMAL
        else:
            inv_multiplied.loc[date, ticker] = inv_base.loc[date, ticker] * MULT_WEAKENING

inv_sums = inv_multiplied.sum(axis=1)
inv_target = inv_multiplied.div(inv_sums, axis=0)
inv_target = inv_target.apply(cap_weights, axis=1)
inv_target = inv_target.mul(regime_pct, axis=0)
inv_weights = inv_target.shift(1).dropna(how='all')
inv_w = inv_weights.loc[common_idx]
inverse_returns = (inv_w * r).sum(axis=1).loc[first_valid:].dropna()
cum_inverse = (1 + inverse_returns).cumprod()

spread = cumulative.iloc[-1] - cum_inverse.iloc[-1]

print(f"\nCumulative returns:")
print(f"  Strategy v8:             {cumulative.iloc[-1] - 1:>8.2%}")
print(f"  Equal-weight benchmark:  {cum_ew.iloc[-1] - 1:>8.2%}")
if has_6040:
    print(f"  60/40 SPY/AGG:           {cum_6040.iloc[-1] - 1:>8.2%}")
print(f"  Inverse momentum:        {cum_inverse.iloc[-1] - 1:>8.2%}")
print(f"\n  Momentum spread: {spread:+.2%} {'(SIGNAL WORKS)' if spread > 0 else '(signal broken)'}")

## Step 6: Performance Table

In [ ]:
def calc_metrics(returns, periods=52):
    n = len(returns)
    total = (1 + returns).prod() - 1
    ann_ret = (1 + total) ** (periods / n) - 1
    ann_vol = returns.std() * np.sqrt(periods)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    cum = (1 + returns).cumprod()
    dd = (cum - cum.expanding().max()) / cum.expanding().max()
    max_dd = dd.min()
    calmar = ann_ret / abs(max_dd) if max_dd != 0 else 0
    win_rate = (returns > 0).mean()
    downside = returns[returns < 0].std() * np.sqrt(periods)
    sortino = ann_ret / downside if downside > 0 else 0
    return {'Total Return': total, 'Ann. Return': ann_ret, 'Ann. Volatility': ann_vol,
            'Sharpe': sharpe, 'Sortino': sortino, 'Max Drawdown': max_dd,
            'Calmar': calmar, 'Win Rate': win_rate}

results = {}
results['Strategy v8'] = calc_metrics(portfolio_returns)
results['Equal-Weight'] = calc_metrics(ew_returns)
if has_6040:
    results['60/40 SPY/AGG'] = calc_metrics(bench_6040)
results['Inverse Mom'] = calc_metrics(inverse_returns)

metrics_df = pd.DataFrame(results).T
display_df = metrics_df.copy()
for c in ['Total Return', 'Ann. Return', 'Ann. Volatility', 'Max Drawdown', 'Win Rate']:
    display_df[c] = display_df[c].map(lambda x: f"{x:.2%}")
for c in ['Sharpe', 'Sortino', 'Calmar']:
    display_df[c] = display_df[c].map(lambda x: f"{x:.2f}")

print("=" * 100)
print("PERFORMANCE COMPARISON")
print("=" * 100)
print(display_df.to_string())

# Quick comparison to v6
print(f"\n--- For reference: v6 had Sharpe 0.77, Total 129.8%, Spread +81.6%, Max DD -13.4% ---")

## Step 7: Visualizations

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]})

ax1 = axes[0]
ax1.plot(cumulative.index, cumulative.values, linewidth=2.5, label='Strategy v8', color='#2E86AB')
ax1.plot(cum_ew.index, cum_ew.values, linewidth=1.5, label='Equal-Weight', linestyle='--', color='#A23B72')
if has_6040:
    ax1.plot(cum_6040.index, cum_6040.values, linewidth=1.5, label='60/40', linestyle=':', color='#F18F01')
ax1.plot(cum_inverse.index, cum_inverse.values, linewidth=1.2, label='Inverse', linestyle='--', color='#999', alpha=0.7)
ax1.set_title('Cumulative Returns', fontsize=16, fontweight='bold')
ax1.set_ylabel('Growth of $1')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
dd = (cumulative - cumulative.expanding().max()) / cumulative.expanding().max()
ax2.fill_between(dd.index, dd.values, 0, alpha=0.3, color='red')
ax2.plot(dd.index, dd.values, linewidth=1, color='darkred')
ax2.set_title('Strategy Drawdown', fontsize=14, fontweight='bold')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Annual returns
ann_strat = portfolio_returns.groupby(portfolio_returns.index.year).apply(lambda x: (1 + x).prod() - 1)
ann_ew = ew_returns.groupby(ew_returns.index.year).apply(lambda x: (1 + x).prod() - 1)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(ann_strat))
width = 0.35
ax.bar(x - width/2, ann_strat.values, width, label='Strategy v8', color='#2E86AB', alpha=0.85)
ax.bar(x + width/2, ann_ew.reindex(ann_strat.index).values, width, label='Equal-Weight', color='#A23B72', alpha=0.85)
for i, (s, e) in enumerate(zip(ann_strat.values, ann_ew.reindex(ann_strat.index).values)):
    if s > e:
        ax.annotate('*', (i - width/2, s + 0.002), ha='center', fontsize=14, color='green', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(ann_strat.index)
ax.set_title('Annual Returns (* = outperformed EW)', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.axhline(y=0, color='black', linewidth=0.8)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

years_won = (ann_strat > ann_ew.reindex(ann_strat.index)).sum()
print(f"Strategy beat equal-weight in {years_won}/{len(ann_strat)} years")

In [ ]:
# Weight heatmap (last 52 weeks)
fig, ax = plt.subplots(figsize=(16, 7))
recent_w = weights.tail(52).T
recent_w.columns = [d.strftime('%m/%d') for d in recent_w.columns]
recent_w.index = [f"{t} ({UNIVERSE[t]})" for t in recent_w.index]
sns.heatmap(recent_w, cmap='YlOrRd', linewidths=0.2, vmin=0, vmax=MAX_WEIGHT, ax=ax,
            cbar_kws={'label': 'Weight'})
ax.set_title('Portfolio Weights — Last 52 Weeks', fontsize=14, fontweight='bold')
ax.set_xticks(ax.get_xticks()[::4])
plt.tight_layout()
plt.show()

In [ ]:
# Contribution analysis
contributions = (w * r).loc[first_valid:]
total_contrib = contributions.sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#D32F2F' if v < 0 else '#2E7D32' for v in total_contrib.values]
ax.barh([f"{t} ({UNIVERSE[t]})" for t in total_contrib.index],
        total_contrib.values, color=colors, alpha=0.85)
ax.set_title('Cumulative Return Contribution by Asset', fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))
ax.axvline(x=0, color='black', linewidth=0.8)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## Step 8: Turnover Analysis

In [ ]:
weekly_turnover = weights.diff().abs().sum(axis=1)

print(f"Turnover Statistics:")
print(f"  Avg weekly turnover: {weekly_turnover.mean():.2%} of portfolio")
print(f"  Median: {weekly_turnover.median():.2%}")
print(f"  Max single-week: {weekly_turnover.max():.2%}")
print(f"  Weeks with no rebalancing: {(weekly_turnover < 0.01).sum()}/{len(weekly_turnover)} ({(weekly_turnover < 0.01).mean():.0%})")
print(f"\n  Avg $ turnover on $50M: ${CAPITAL * weekly_turnover.mean():,.0f}/week")

## Step 9: Current Portfolio (for IBKR)

In [ ]:
sig_date = weights.index[-1]
cur_weights = weights.loc[sig_date].sort_values(ascending=False)
gld_1m = gld_trail.loc[sig_date] if sig_date in gld_trail.index else 0
tlt_1m = tlt_trail.loc[sig_date] if sig_date in tlt_trail.index else 0

# Determine regime
neg_count = int((gld_1m < 0) + (tlt_1m < 0))
regime_label = ['FULL', 'CAUTION', 'STRESS'][neg_count]

print("=" * 75)
print(f"PORTFOLIO FOR NEXT WEEK — Signal: {sig_date.date()}")
print("=" * 75)
print(f"\nRegime: {regime_label} ({cur_weights.sum():.0%})")
print(f"  GLD 1M: {gld_1m:.2%}  |  TLT 1M: {tlt_1m:.2%}")

print(f"\n{'Rank':<5} {'Ticker':<7} {'Asset':<25} {'12-1M Ret':>10} {'EMA':>5} {'Weight':>8} {'$ Alloc':>14}")
print("-" * 78)

for i, (t, wt) in enumerate(cur_weights.items(), 1):
    rank = int(latest_rank[t])
    ema_ok = 'YES' if latest_ema.get(t, False) else 'NO'
    mom = latest_mom.get(t, 0)
    dollar = CAPITAL * wt
    print(f"{i:<5} {t:<7} {UNIVERSE[t]:<25} {mom:>+9.2%} {ema_ok:>5} {wt:>7.1%} ${dollar:>13,.0f}")

print(f"\nTotal invested:                              {cur_weights.sum():.1%} (${CAPITAL * cur_weights.sum():>13,.0f})")
print(f"Cash:                                        {1-cur_weights.sum():.1%} (${CAPITAL * (1-cur_weights.sum()):>13,.0f})")

In [ ]:
# Save outputs
# Uncomment locally to generate CSV files (excluded from repo via .gitignore)
# weights.to_csv('weight_history_v8.csv')
# all_returns = pd.DataFrame({'Strategy': portfolio_returns, 'EW_Benchmark': ew_returns, 'Inverse': inverse_returns})
# if has_6040:
#     all_returns['Benchmark_6040'] = bench_6040
# all_returns.to_csv('returns_history_v8.csv')

# Save performance summary to results/
import os
os.makedirs('results', exist_ok=True)
metrics_df.to_csv('results/performance_summary.csv')
print("Saved: results/performance_summary.csv")
print("Note: weight_history_v8.csv and returns_history_v8.csv are excluded from repo.")
print("Run the notebook locally to regenerate them.")


In [ ]:
# ============================================================
# SUB-PERIOD PERFORMANCE (Professor feedback: "slice into independent time periods")
# ============================================================

periods = {
    '2013-2015': ('2013-01-01', '2015-12-31'),
    '2016-2018': ('2016-01-01', '2018-12-31'),
    '2019-2021': ('2019-01-01', '2021-12-31'),
    '2022-2024': ('2022-01-01', '2024-12-31'),
    '2025-2026': ('2025-01-01', '2026-12-31'),
}

print("=" * 85)
print("SUB-PERIOD PERFORMANCE (independent time slices)")
print("=" * 85)
print(f"{'Period':<12} {'Strat Return':>13} {'EW Return':>11} {'Strat Sharpe':>13} {'EW Sharpe':>10} {'Spread':>8} {'Winner':>8}")
print("-" * 85)

strat_wins = 0
for label, (start, end) in periods.items():
    mask = (portfolio_returns.index >= start) & (portfolio_returns.index <= end)
    sr = portfolio_returns[mask]
    er = ew_returns.reindex(sr.index).dropna()
    
    if len(sr) < 10:
        continue
    
    s_total = (1 + sr).prod() - 1
    e_total = (1 + er).prod() - 1
    s_sharpe = (sr.mean() / sr.std()) * (52 ** 0.5) if sr.std() > 0 else 0
    e_sharpe = (er.mean() / er.std()) * (52 ** 0.5) if er.std() > 0 else 0
    spread = s_total - e_total
    winner = "STRAT" if s_total > e_total else "EW"
    if s_total > e_total:
        strat_wins += 1
    
    print(f"{label:<12} {s_total:>+12.2%} {e_total:>+10.2%} {s_sharpe:>13.2f} {e_sharpe:>10.2f} {spread:>+7.2%} {winner:>8}")

print(f"\nStrategy won {strat_wins}/{len(periods)} sub-periods")

In [ ]:
# ============================================================
# ABLATION TEST: isolating contribution of each component
# ============================================================

# We test 4 variants against the same return data:
# 1. Full v8 (STRONG 1.3x + EMA 0.5x + regime filter) — already computed
# 2. No EMA filter: all assets get pure rank-weight, no penalty
# 3. No STRONG boost: EMA filter at 0.5x but no 1.3x for top 5
# 4. No regime filter: always 96% invested

common_dates = portfolio_returns.index

# --- Variant 2: No EMA filter (pure rank-weight) ---
no_ema_weights = base_weights.copy()
no_ema_weights = no_ema_weights.apply(cap_weights, axis=1)
no_ema_weights = no_ema_weights.mul(regime_pct, axis=0)
no_ema_weights = no_ema_weights.shift(1).dropna(how='all')
no_ema_r = (no_ema_weights.loc[common_dates] * r.loc[common_dates]).sum(axis=1)

# --- Variant 3: EMA filter but no STRONG boost (just 1.0x and 0.5x) ---
no_boost = base_weights.copy()
for date in momentum.index:
    for ticker in tickers:
        ema_ok = ema_confirmed.loc[date].get(ticker, False)
        if not ema_ok:
            no_boost.loc[date, ticker] = base_weights.loc[date, ticker] * MULT_WEAKENING
        # else keep at 1.0x (no STRONG boost)
nb_sums = no_boost.sum(axis=1)
no_boost = no_boost.div(nb_sums, axis=0)
no_boost = no_boost.apply(cap_weights, axis=1)
no_boost = no_boost.mul(regime_pct, axis=0)
no_boost = no_boost.shift(1).dropna(how='all')
no_boost_r = (no_boost.loc[common_dates] * r.loc[common_dates]).sum(axis=1)

# --- Variant 4: No regime filter (always 96%) ---
no_regime_weights = target_weights.copy()
no_regime_sums = no_regime_weights.sum(axis=1)
no_regime_weights = no_regime_weights.div(no_regime_sums, axis=0) * 0.96
no_regime_weights = no_regime_weights.shift(1).dropna(how='all')
# Apply same turnover buffer
no_regime_buf = no_regime_weights.copy()
for i in range(1, len(no_regime_buf)):
    prev = no_regime_buf.iloc[i-1]
    curr = no_regime_weights.iloc[i]
    delta = (curr - prev).abs()
    no_regime_buf.iloc[i] = np.where(delta < TURNOVER_BUFFER, prev, curr)
    row_sum = no_regime_buf.iloc[i].sum()
    if row_sum > 0:
        no_regime_buf.iloc[i] = no_regime_buf.iloc[i] / row_sum * 0.96
no_regime_r = (no_regime_buf.loc[common_dates] * r.loc[common_dates]).sum(axis=1)

# --- Results ---
def quick_metrics(ret):
    total = (1 + ret).prod() - 1
    ann = (1 + total) ** (52 / len(ret)) - 1
    vol = ret.std() * (52 ** 0.5)
    sharpe = ann / vol if vol > 0 else 0
    cum = (1 + ret).cumprod()
    dd = ((cum - cum.expanding().max()) / cum.expanding().max()).min()
    return total, ann, sharpe, dd

variants = {
    'Full v8 (final model)': portfolio_returns,
    'No EMA filter': no_ema_r,
    'No STRONG boost': no_boost_r,
    'No regime filter': no_regime_r,
    'Equal-weight baseline': ew_returns,
}

print("=" * 90)
print("ABLATION TEST: contribution of each component")
print("=" * 90)
print(f"{'Variant':<25} {'Total Return':>13} {'Ann Return':>11} {'Sharpe':>8} {'Max DD':>9}")
print("-" * 70)

for name, ret in variants.items():
    total, ann, sharpe, dd = quick_metrics(ret)
    marker = " ◄ FINAL" if name == 'Full v8 (final model)' else ""
    print(f"{name:<25} {total:>+12.2%} {ann:>+10.2%} {sharpe:>8.2f} {dd:>+8.2%}{marker}")

print("\nInterpretation:")
full_sharpe = quick_metrics(portfolio_returns)[2]
no_ema_sharpe = quick_metrics(no_ema_r)[2]
no_boost_sharpe = quick_metrics(no_boost_r)[2]
no_regime_sharpe = quick_metrics(no_regime_r)[2]

print(f"  EMA filter adds:    {full_sharpe - no_ema_sharpe:+.3f} Sharpe")
print(f"  STRONG boost adds:  {full_sharpe - no_boost_sharpe:+.3f} Sharpe")
print(f"  Regime filter adds: {full_sharpe - no_regime_sharpe:+.3f} Sharpe")